# Практика · TF-IDF

> Лекція: [lecture.html](lecture.html) · Домашнє завдання: [homework.md](homework.md) ·
> Тест: [quiz.html](quiz.html)

Тут ми рахуємо **всі** числа, які називає лекція. Порядок такий самий, як у ній.

Що зробимо:

1. Зберемо корпус українських перекладів інтерфейсів і подивимось на нього очима.
2. Порахуємо IDF і знайдемо слова з найменшою й найбільшою вагою.
3. Перевіримо головне твердження теми: чи змінює IDF те, яке слово в документі головне.
4. Напишемо TF-IDF **своїми руками** і звіримо із `sklearn` до чотирнадцятого знака.
5. Заміряємо, що буває без логарифма, без згладжування і без нормалізації.
6. Зламаємо TF-IDF трьома різними способами — і кожен замір покажемо числом.
7. Зберемо підсумок усього блоку 1 в одну таблицю.

> ⏱ Заміряно: близько **50 секунд** на чотирьох ядрах без відеокарти.
> Найдовше йде тричі повторений пошук по 20 000 документів.

## 1 · Середовище

Перша клітинка друкує версії. Якщо в тебе інші — числа можуть трохи поїхати,
і краще знати про це одразу, а не наприкінці.

In [ ]:
import sys, re, math, glob, gettext, time, collections
import numpy as np
import scipy.sparse as sp
import sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize

print("Python  ", sys.version.split()[0])
print("numpy   ", np.__version__)
print("scipy   ", __import__("scipy").__version__)
print("sklearn ", sklearn.__version__)

## 2 · Корпус: українські переклади інтерфейсів

Кожна програма в Linux має файл перекладу `.mo`: там лежать пари «англійський
оригінал → український переклад». Ми беремо **переклади** як документи.

Чому саме цей корпус: це справжня українська мова, він лежить на диску (жодної
мережі) і не міняється між запусками. Чому він **не** універсальний: це вузький
домен — технічна лексика, короткі речення, багато наказового способу. Висновків
про мову взагалі з нього робити не можна, і далі ми побачимо, як саме це вилазить.

Якщо української локалі на машині немає, вмикається вбудований запасний корпус —
маленький, але достатній, щоб увесь зошит виконався до кінця.

In [ ]:
def load_system_corpus():
    """Читаємо всі .mo-файли української локалі. Повертаємо трійки
    (програма, англійський оригінал, український переклад)."""
    docs = []
    for path in sorted(glob.glob('/usr/share/locale/uk/LC_MESSAGES/*.mo')):
        try:
            with open(path, 'rb') as f:
                catalog = gettext.GNUTranslations(f)
        except Exception:
            continue                       # зламаний або чужий формат — просто пропускаємо
        program = path.split('/')[-1][:-3]
        for src, dst in catalog._catalog.items():
            # службовий заголовок каталогу має ключ '' і не є текстом
            if isinstance(src, str) and isinstance(dst, str) \
               and len(dst) > 30 and 'Project-Id' not in dst:
                docs.append((program, src, dst))
    return docs

FALLBACK = [
": попередження: не вдалося отримати робочий каталог",
"запакувати вільні недосяжні обʼєкти",
"Створення зірок та багатокутників",
"У пристрою шифрування не вистачає альтернативної назви",
"Ініціалізатор компонента без імені після компонента з іменем в",
"Програмі перегляду потрібен термінал",
"Телефон виклику служб надзвичайних ситуацій",
"ключ : некоректний безпосередній підпис ключа",
"для типу немає доступної функції виводу",
"розіменувати теги в ідентифікатори обʼєктів",
"скопіювати файли з іменованої стадії",
"Перевірити, чи існує вказаний елемент або додаток",
"домен не перебуває у стані призупинки",
"для згрупованих цілей має бути надано спосіб збирання",
"Перетворити умовні збереження в безумовні.",
"встановити назву файла для виходу налагоджування",
"декілька класів зберігання в специфікаторах оголошення",
"Вибрати файл або каталог для встановлення мітки.",
"Пароль можна підібрати за словником",
"Постфіксні оператори не підтримуються.",
"не вдалося отримати параметри за замовчуванням",
"Базу даних піктограм ще не ініціалізовано",
"несумісний тип для методу",
"немає розрідженого переходу для додавання",
"Сталася помилка під час обробки параметра :",
"Взяти кольори для позначених кутових вузлів із базової сітки.",
"Попередній перегляд експозиції режиму інтерактивного перегляду",
"Інструкції з повідомлення про помилку дивіться за посиланням:",
"Деякі сховища не було оновлено через помилку.",
"Ймовірно, вам варто змінити також і його строк дії.",
"Під час налаштування обробника сигналів:",
"Перезавантаження системи потрібне для:",
"не вдалося прочитати дані розділу:",
"Передчасне закінчення регулярного виразу",
"видалити віддалені призначення після отримання",
"помилка при обчисленні нового контексту",
"не є іменем відомого системі користувача",
"Тестовий параметр лише для читання",
"Виводити більш докладні описи потоку керування в діагностиці.",
"Такої команди не знайдено, коректні команди:",
"для потрібно вказати некваліфіковані імена відносин",
"Очікуване імʼя звʼязування в",
"не вдалося створити робочий потік:",
"Невідома піктограма для сторінки",
"Надіслати колекцію латок у вигляді електронних листів",
"видати тільки вивід, що відноситься до другого діапазону",
"Увімкнути інструкції ділення та залишку.",
"байтів тіла все ще очікуються",
"некоректна назва векторного регістра",
"Увімкнути пришвидшення обробки плоских полотен",
"Не вдалося створити відгалуження:",
"Не вдалося повторно створити початкове місце:",
"недійсні операнди в бінарній операції",
"Проблеми з відкриттям пакунка",
"код операції ні на що не вплине",
"Неочікуваний передчасний кінець потоку",
"адреса з постінкрементом не є регістром",
"Отримати назву батьківського знімка",
"Отримує налаштовані віддалені пристрої",
"не вдалося завантажити список блокування:",
"Не вдалося записати файл обʼєкта",
"Валюта, в яку перетворюється поточні розрахунки",
"параметри містив пароль з порожньою назвою",
"Буде встановлено при наступному вході",
"перемістити файли у непридатному до переміщення пакунку",
"індекс доріжки повинен бути константою негайним",
"не вдалося створити тимчасовий каталог:",
"Цифровий трансфокатор не використовувався",
"Наступні параметри специфічні для цілей",
"Відсутня початкова ліва дужка у рядку форматування на",
"Показати панель налаштовування позначеного способу введення",
"Не вдалося перевірити підпис:",
"Надіслати налагоджувальний вивід у вказаний файл",
"Дозволити доступ до буфера обміну",
"спорожнити файл уповноважених ключів до додавання нових ключів",
"Помилка запису у сеансі :",
"Чи показується розмір вибраного шрифту у позначці",
"перевіряти адреси сегментів на перекриття",
"Креольські діалекти на основі португальської",
"Рівень заряду акумулятора пристрою є надто низьким",
"мета статистики знижується до",
"Слід вказати шлях до бази даних",
"пропущено: закритий ключ вже існує",
"Вивід більш докладної інформації.",
"не вдалося отримати назву розділу",
"не вдалося отримати поточний робочий каталог",
"Увімкнути консервативне розгортання невеликих циклів.",
"Останній розділ за порядком читання",
"Назва програми, як вона використовується менеджером вікон",
"Бажана дія є некоректною через свою невизначеність",
"занадто багато відкриваючих дужок",
"у повідомлення не залишилось даних",
"Розмір , але файлова система .",
"не вдалося знайти файл скрипту",
"Чи впливатиме цей тег на колір переднього плану",
"Додати вміст файлу до області індексації",
"спроба розмістити дані у абсолютному розділі",
"Перетворення на чорний або білий",
"Спробуйте інші критерії пошуку.",
"Помилка під час виконання команди",
"для тригери подій не підтримуються",
"Помилка під час спроби обробити параметри:",
"Текстура шовкового килима, горизонтальні смужки",
"скаляр поза припустимим діапазоном у інструкції множення",
"Ефект підсвіченого кольорового скла",
"змінити або отримати сталі налаштування",
"не вдалося прочитати заголовок архіву",
"Адреса таблиці ключів для перевірки реєстраційних даних",
"Ханти-Мансійський автономний округ-Югра",
"не вдалося закрити файл журналу",
"Потрібні права, щоб опанувати процеси інших користувачів",
"запис лічильника простих блоків",
"не вдалося записати новий конфігураційний файл",
"Ввести інформацію про розширення покроково",
"Може отримувати доступ до певних файлів",
"створити буфер на основі набору аргументів",
"показати список буферів з увімкненим автоматичним запуском",
"мало бути вказано регістр цілих чисел",
"зупинити роботу домену після створення знімка",
"Використовувати лише апаратні значення",
"мебібайт,мебібайти,мебібайтів,МіБ",
"встановлення ввічливого режиму змінюваних областей",
"Калібрувати фокусування лінз у області документа",
"занадто довгий список аргументів",
"Змінити масштаб, щоб розмістити сторінку по ширині",
"Сталася помилка ініціалізації системи подій",
"некоректний символ у мнемосхемі",
"відновити початкове значення параметру виконання",
"Простір, що додається до елемента керування зліва.",
"інформація про фіксовану версії",
"неможливо двічі змінити параметр стійкості",
"Символи запису музики знаменного співу",
"сміття після числового літерала",
"не вдалося оновити заголовок розділу:",
"Вивантаження серверного модуля розмітки.",
"частка стовпчика не є числом:",
"Векторизація за даними користувача",
"Рівень пунктуації вказано для більшості.",
"Чи показувати завантажувані місця",
"перетворення припинено через проблеми із записом результату",
"Інверсивне зображення у чорно-білому або півтоновому режимі",
"Горно-Бадахшанська автономна область",
"Вкажіть потрібну вам еліптичну криву:",
"аргумент, який може бути виділений",
"Завершення роботи за бажанням користувача",
"Початкова точка лінії віддзеркалення",
"неможливо додати контракти до віртуальної функції",
"вказані ключі для двох різних операцій",
"Спорожнити резервні копії позначення",
"виклик функції має агрегатне значення",
"контекст мітки не є поточною декларацією функції",
"потребує додаткові драйвера до принтера.",
"Можна відновити скорочення, натиснувши .",
"недійсне порівняння непорівняльного масиву",
"не вдалося ініціалізувати сервер:",
"подвійні константи не підтримуються",
"тимчасові домени не мають будь-яких сталих налаштувань",
"Перемістити вікно на монітор праворуч",
"стан інтерфейсу керування домену",
"Сховище є безпечним для оприлюднення:",
"Перейти на шар, що знаходиться нижче від поточного",
"Не вдалося створити тимчасовий файл кешу",
"некоректна точність у форматуванні",
"Оновлення контролера сховища даних",
"Рядок посилається на аргумент з номером несумісним способом.",
": спроба обертання регістра лічильника команд",
"Подвійна мітка вказівки на і",
"Увімкнути напрямну клавішу для вибору варіанта",
"елементи керування відтворенням та показом стану аудіо",
"відносний виклик поза межами припустимого діапазону",
": не допускається створювати жорсткі посилання на каталоги",
"Зміна розмірів зашифрованого пристрою",
"Налаштуйте для вказаного цільового процесора або архітектури.",
"атрибути повернуто у некоректному порядку",
"Не вдалося відкрити файл закритого ключа :",
"не вдалося отримати назву додатка до редактора:",
"Додаткові пункти контекстного меню",
"Дозволити спрощення типів кольорів",
"Отримати дані щодо розміру блокового пристрою для домену.",
"Не вдалося обробити список сеансів",
"повторно використати вказаний обʼєкт нотатки",
"не вдалося ввімкнути технічне обслуговування",
"визнати вказаний суфікс як імʼя файлу модуля визначення",
"вказано номер операнда для формату, який не приймає аргументів",
"Перевірити, чи є сертифікат сервера журналювання чинним",
"Користувацькі або вбудовані змінні функції не допускаються.",
"Не вдається визначити форму ініціалізатора на",
"кількість секунд до завершення відтворення відео",
"операнд розгалуження не вирівняно",
"змінна перелічування в швидкому перелічуванні не є обʼєктом",
"Повернути за годинниковою стрілкою",
"Кількість пробілів на рівень відступу:",
"Комбінація клавіш для вмикання або вимикання меню параметрів.",
"Незавершена символьна константа, що починається на",
"Не вдалося створити анонімну трубу",
"Потрібно утримувати пристрій на екрані протягом калібрування.",
"Низький заряд батареї клавіатури",
"застосування до порожньої історії",
"Не вдалося показати перелік класів",
"Пікселів додаткового простору на початку",
"Збільшити відстань між літерами",
"Помилка під час спроби обробити файл:",
"Цілунки між людьми однієї статі",
"Обмежити доступ до файлової системи",
"помилка вводу-виводу у потоці даних",
"Синтаксична помилка в операторі на",
"Не вдалося оновити дані щодо розміру файлової системи:",
"Не вдалося опитати список читачів:",
"Вибрати колір для варіантів з бази даних користувача",
"мегабайт,мегабайти,мегабайтів,МБ",
"Гарнітура шрифту для індикатора розкладки",
"додати до схову тільки індексовані зміни",
"Встановити назву файла вихідних даних",
"не надіслав всіх необхідних обʼєктів",
"Не вдалося прочитати повноваження сокета:",
"Не вдалося створити простий параметр",
"Імітувати натискання послідовності клавіш",
"розпочати проходження по всім посиланням",
"Пакунки зі зниженням версії:",
"Не вдалося повністю кешувати ресурс",
"Спробувати знову отримати файл.",
"Порівняти версії, які вказано як аргументи.",
"НАЗВА РОЗТАШУВАННЯ - Додати віддалене сховище",
": некоректне визначення часу очікування",
"Програма хоче захоплювати події з введення даних",
"Південні народи, національності і люди",
"Некоректне пересування для поля",
"Вивести результат зі зворотним порядком байтів",
"запустити домен під час запуску",
"Список функцій підтримки сімейств операторів",
"Вторинний сертифікат не містить відкритого ключа",
"номер кімнати нового користувача",
"Допоміжний заголовок стовпчика таблиці",
"Неможливо поставити в чергу і замінити одночасно",
"Не забезпечено методів розпізнавання",
"шлях до файла з заголовком пароля",
"Не вдається записати блок розділу у .",
"Кількість спроб введення пароля:",
"ця збірка не підтримує стиснення з",
"дані щодо власника блокування не було зареєстровано",
"Опитати щодо підтримки оновлення мікропрограми",
"недоступний обліковий запис користувача",
"Цю інструкцію не можна використовувати на цій архітектурі",
"Уникати показування списку користувачів",
"вказаний розмір може перевищувати максимальний розмір обʼєкта",
"Немає операцій, що можна повернути.",
"вимкнути взяття у лапки вказаних символів",
"завершився дочірній процес , очікувалося",
"високосна секунда до початку епохи",
"Створення, зміна архівів і видобування даних з архівів.",
": : помилка під час спроби завантажити додаток:",
"Ваші локальні зміни у наступних файлах буде замінено злиттям:",
"Підрозділів на основну кутову поділку:",
"Без азартних ігор у будь-яких її проявах",
"Не вдається перетворити назву файлу",
"Виконати вибіркове планування після перезавантаження.",
"Не перевіряти або не встановлювати залежності для виконання",
"Не вдалося отримати дані щодо приймача:",
"неможливо відкрити вхідний файл",
"шматки не перетинаються: не закінчується на:",
"Короткий опис того, що розширення робить",
"Степінь збільшення розміру елемента по вертикалі",
": латка не може бути застосована",
"Чи впливає цей тег на підкреслення",
"домен позначено для автоматичного знищення",
"Програми, що не належать до жодної категорії",
"Ефект шовковистої алюмінієвої поверхні з рельєфністю",
"Гуансі-Чжуанський автономний район",
"не дозволяється безіменне перелічування з областю видимості",
"регістри призначення і джерела мають бути різними",
"Наступний вузол за порядком читання вузлів",
"має бути розташовано на пристрої",
"виберіть режим стискання для основного образу",
"Скоригувати розмір піктограми на панелі інструментів",
"Вирівняти позначене горизонтально за лівим краєм",
"частота обертання має сенс лише для дискових пристроїв",
"Вивести список компонентів, які є частиною вказаних категорій.",
"Застарівання лексеми розпізнавання вимкнено",
"Вказано невідомий алгоритм або протокол.",
"Носій не має ідентифікатора, неможливо вилучити",
"Гірський час - Байя, Наяріт, Сіналоа",
"Вказівник на піксельні дані буфера пікселів",
"кількість процесорів є надто великою",
"Повідомлення про помилку від сервера:",
"неможливо виконати резервне копіювання для неактивного домену",
"Підтримки пошуку за адресою електронної пошти не передбачено",
"Без автоматичного керування підсиленням",
"Програмні засоби для цього пристрою не встановлено.",
"Перейти до робочого простору праворуч",
"Параметр повинен використовуватись сам по собі.",
"Прилипання лише до вузла, найближчого до вказівника",
"Змінна ітерації використовується в більш ніж одній петлі на",
"помилка під час оновлення робочої директорії",
"не вказано архітектурного розширення",
"Замінити значення оболонки від надавача профілю цим значенням",
"Переглянути детальний журнал подій для системи",
"неназваний тимчасовий визначений тут",
"використовувати глобальний файл конфігурації",
"запит щодо пароля до ключа для підписування",
"Помилка при записуванні у потік кешу",
"Скоро сеанс буде покинуто через тривалу бездіяльність.",
"Помилка: не вказано назви методу",
"Файл не є звичайним файлом.",
"Не вдалося передати дані в вікно",
"контролювати рекурсивне отримання підмодулів",
"не розкодовувати назви символів",
"батьківська підпрограма не може бути вбудована",
"отримати дані щодо активного завдання для вказаного диска",
"вираз-оператор в константному виразі",
"Не вдалося почати прослуховування за адресою , порт :",
"Назви файлів не мають закінчуватись пропуском",
"Вимикати сенсорний пристрій при наборі на клавіатурі",
"функція вводу типу повинна повертати тип",
"рекурсивно у вкладених підмодулях",
"Перевірте, що диск вставлений у пристрій.",
"Виданий сертифікат ще не дійсний.",
"Показує розмір дискового блоку.",
"Виберіть місце, яке слід ігнорувати",
"Будь ласка, введіть відомості щодо вашого вторинного ключа:",
"Не вдалося завершити побудову ієрархічного списку",
"Помилка при вступі до мультикастової групи:",
"Дані на буде перезаписано без можливості відновлення.",
"Попереджувати, якщо використовується масив змінної довжини.",
"Не вдалося знайти пересування для інструкції",
"не архівувати каталоги систем керування версіями",
"не вдається змінити поточний каталог",
"отримати рядки з таблиці або подання",
"Не вдалося прочитати вхідні дані користувача",
"Виправлення для проблеми номер не знайдено або не потрібне.",
"Некоректне перехресне посилання на пристрій",
": не вдалося розпізнати формат файла",
"Попереджати про виклики з неявним інтерфейсом.",
"Неправильний символ табуляції в стовпці рядка",
"Граничний час між двома перевірками для поновлення",
"показати лише встановлені пакунки",
"одночасно два правила для одного випадку",
"Ширина переносу для розташування елементів сітки",
"помилка при встановленні нової ролі:",
"показати розширені параметри томів",
": підтримки пересування опосередкованої функції не передбачено",
"Рядок, який буде використано для некоректних символів",
"використання стеку може бути необмеженим",
"Виберіть словник із найвищою пріоритетністю",
"Дозволити швидкі переходи до диспетчера повідомлень.",
"схожий на номер державного страхування.",
"Обсяг простору, що займає стрілка",
"рядок : : пакунок вже існує",
"додати більше причин тайм-ауту не можна",
"Режимом розбиття має бути",
"Виникли проблеми із запитом щодо встановлення модуля:",
"Ігноруємо адресу без структури і назви вузла",
"Не вдалося виконати читання з :",
"Вимовляти координати комірок таблиці",
"непотрібне імʼя типу в порожньому оголошенні",
"не вдалося встановити таймер:",
"Не вдалося додати апаратну частину машини",
"мало бути використано попередньо індексований вираз",
"Додавання нового коментаря до запису вади",
"не вказано тип основного пристрою",
"Режим, наприклад окрайок або фаска",
"Використовувати природне значення висоти актора",
"Спробувати перевірити команду і аргументи після виконання",
"Виберіть типи файлів, які слід показувати",
"Шлях імпортування має вказувати на наявне сховище даних.",
"неможливо переіндексувати вказані таблиці в усіх базах даних",
"Дозволити навігацію до адрес із даними з верхнього фрейма",
"аргумент для масиву зі змінною довжиною занадто великий",
"глобальні деструктори не підтримуються на цій платформі",
"Виберіть пристрій або файл носія ОС",
"Встановленими пакунками надано декілька платформ модулів",
"показати сторінку керівництва користувача",
"Типовий масштаб при перегляді піктограм",
"директорія призначення не існує",
"Файли, до яких позначений домен може записувати дані.",
"Ефект бульбашки з заломленням і сяйвом",
"Чи цей тег впливає на ліву межу",
"Картатий візерунок з широкими можливостями налаштування",
"Розмір у точках, що використовується для значка з назвою",
"Помилка під час виконання ітерації списком блоків каталогів:",
"Стара версія програмного інтерфейсу",
"Пропустити поточне вікно під час пошуку",
"останній аргумент повинен бути негайним значенням",
"Вибирати тему кнопок зі стрілками у вікні варіантів",
"профіль петлі не може бути портом",
"Не вдалось створити файл під , оскільки це не є текою",
": Помилка при читанні заголовка файла",
"Систему подій не ініціалізовано",
"Затримка перед появою підменю панелей меню",
"Деструктор не повертає значення для перевірки",
"Не вдалося закрити обробку файла",
"Спосіб пересування звичайних операндів типу є невідомим",
"Не вдалося створити символічне посилання під час перенесення :",
"суперечливі параметри визначення ширини",
"Вибраний вами файл є каталогом:",
"Некоректна послідовність байтів у вхідних даних перетворення",
"... : кілька баз злиття, використання",
"Підтримка користування на малому екрані",
"Вертикальні альтернативи для обертання",
"Цільовий кластер завершив роботу некоректно.",
"Не використовуйте апаратне з плаваючою комою."
]

corpus = load_system_corpus()
if len(corpus) < 5000:
    # української локалі на машині немає — працюємо на вбудованому корпусі
    corpus = [('fallback', '', t) for t in FALLBACK]
    print("⚠️  системної локалі немає, працюємо на вбудованому корпусі")
else:
    print("шлях спрацював: /usr/share/locale/uk/LC_MESSAGES/*.mo")

programs = sorted(set(p for p, _, _ in corpus))
print("документів:", len(corpus))
print("програм:   ", len(programs))
print()
for program, source, target in corpus[:3]:
    print(f"[{program}] {source[:46]!r}\n         -> {target[:60]!r}")

## 3 · Що ми вважаємо словом

`CountVectorizer` і `TfidfVectorizer` за замовчуванням ріжуть текст правилом
`\b\w\w+\b`: послідовності з **двох і більше** «словесних» символів. Для української
це погано з двох причин одразу: викидає односимвольні прийменники «у», «з», «і» —
а це найчастіші слова мови, — і залишає цифри.

Тому ми задаємо своє правило: одна або більше **літер**, без цифр і підкреслень.

Друге рішення: прибираємо підстановки формату (`%s`, `%d`, `{0}`). Це не слова мови,
а дірки, куди програма вставить імʼя файла чи число. Якщо їх лишити, буква `s` із
`%s` стає сьомим за частотою «словом» корпусу.

In [ ]:
# %s, %(name)d, {0}, $1, &amp; — це не слова, а місця під підстановку
PLACEHOLDER = re.compile(r"%\([^)]*\)?[a-zA-Z]|%[-+ #0-9.*']*[a-zA-Z%]|\{[^{}]*\}|\$\d|&[a-z]+;")

# одне й більше літер: лишає «у» і «з», викидає цифри й підкреслення
TOKEN_PATTERN = r"(?u)\b[^\W\d_]+\b"
tokenize = re.compile(TOKEN_PATTERN).findall

def clean(text):
    return PLACEHOLDER.sub(" ", text)

documents = [clean(target) for _, _, target in corpus]

example = corpus[17][2]
print("сирий текст :", example)
print("після чистки:", clean(example))
print("токени      :", tokenize(clean(example).lower()))

## 4 · Двадцять тисяч документів і три зерна

Повний корпус великий, а нам треба, щоб зошит виконувався за хвилину. Беремо
**20 000 випадкових документів** — і робимо це **тричі**, з зернами 0, 1 і 2.

Навіщо три зерна: далі ми будемо порівнювати способи зважування, і різниця між
ними має сенс тільки тоді, коли вона більша за розкид між зернами. Різниця, менша
за розкид, — це не різниця, а шум вибірки.

In [ ]:
SEEDS = (0, 1, 2)
N_DOCS = 20000

def sample_documents(seed, n=N_DOCS):
    """Ті самі 20 000 документів при тому самому зерні — на будь-якій машині."""
    rng = np.random.default_rng(seed)
    picked = rng.choice(len(documents), min(n, len(documents)), replace=False)
    return [documents[i] for i in picked]

def count_matrix(texts):
    """Мішок слів із теми 04: рядок — документ, колонка — слово, клітинка — скільки разів."""
    vectorizer = CountVectorizer(token_pattern=TOKEN_PATTERN)
    counts = vectorizer.fit_transform(texts).tocsr()
    return vectorizer, counts

for seed in SEEDS:
    texts = sample_documents(seed)
    vec, counts = count_matrix(texts)
    cells = counts.shape[0] * counts.shape[1]
    print(f"зерно {seed}: матриця {counts.shape[0]} x {counts.shape[1]}, "
          f"ненульових {counts.nnz}, заповнено {100*counts.nnz/cells:.4f} %")

Заповнено чотири соті відсотка — це та сама розрідженість, яку тема 04 назвала
причиною, а не оптимізацією. Тримай це число в голові: усе, що ми робимо далі,
міняє **значення** в цих клітинках, але не їхню кількість.

## 5 · IDF: хто важить мало, а хто багато

`TfidfVectorizer` кладе готові ваги в `idf_`. Формула `sklearn` така:

    idf(w) = ln((1 + N) / (1 + df(w))) + 1

де `N` — скільки всього документів, `df(w)` — у скількох із них слово трапилось
хоч раз. Одиниці й логарифм ми розберемо через два розділи; поки що просто
подивимось, кому скільки дісталося.

In [ ]:
def fit_tfidf(texts, **kwargs):
    vectorizer = TfidfVectorizer(token_pattern=TOKEN_PATTERN, **kwargs)
    matrix = vectorizer.fit_transform(texts).tocsr()
    return vectorizer, matrix

for seed in SEEDS:
    texts = sample_documents(seed)
    tfidf, _ = fit_tfidf(texts)
    words = tfidf.get_feature_names_out()
    idf = tfidf.idf_
    order = np.argsort(idf)
    lowest = ", ".join(f"{words[i]} {idf[i]:.2f}" for i in order[:5])
    at_top = 100 * np.mean(idf == idf.max())
    print(f"зерно {seed}: найнижчий IDF -> {lowest}")
    print(f"          найвищий IDF {idf.max():.2f}, і його має {at_top:.1f} % словника")

Найдешевші слова корпусу — «не», «для», «у», «з». Це очікувано: службові слова є
скрізь, і саме їх IDF мусить придушити.

А от **«вдалося»** серед найдешевших — не очікувано. Подивимось на нього окремо.

In [ ]:
frequency = collections.Counter()
for text in documents:
    frequency.update(tokenize(text.lower()))

print("найчастіші слова всього корпусу:")
for rank, (word, times) in enumerate(frequency.most_common(8), start=1):
    print(f"  {rank}. {word:<10} {times:>6} ужитків")

print()
print("усього слововживань:", sum(frequency.values()))
print("різних словоформ:   ", len(frequency))

«Вдалося» — на пʼятому місці за частотою в усьому корпусі. У звичайній українській
воно ніде близько до пʼятірки не стоїть.

Причина проста: наш корпус зібрано з перекладів інтерфейсів, а інтерфейси більшу
частину часу повідомляють про **невдачі**: «не вдалося відкрити», «не вдалося
зберегти», «не вдалося зʼєднатись». Тема 02 показала те саме з іншого боку: її
токенізатор розрізав «телефон» на чотири шматки, а «налаштування» лишив цілим.
Обидва заміри кажуть одне: **модель знає домен, а не мову**.

## 6 · Головне питання теми: чи змінює IDF відповідь

Питання формулюємо так. Візьмемо документ і спитаємо, яке слово в ньому головне.
Дві відповіді: за самою частотою (мішок слів із теми 04) і за TF-IDF. Чи це те саме
слово?

Тут є пастка, у яку легко впасти. Наші документи короткі, і в переважній більшості
**всі слова трапляються рівно по разу**. У такому документі «найчастіше слово» не
існує: частота не має думки взагалі, і `argmax` поверне просто перше слово за
абеткою. Порівнювати з ним безглуздо.

Тому рахуємо чесно й окремо:

* у скількох документах у частоти взагалі є думка (унікальний максимум);
* серед **тільки цих** документів — у скількох IDF цю думку перекриває.

In [ ]:
def main_word_disagreement(texts):
    vec, counts = count_matrix(texts)
    tfidf, weights = fit_tfidf(texts, vocabulary=vec.vocabulary_)
    silent = unique = overridden = 0
    for row in range(counts.shape[0]):
        start, stop = counts.indptr[row], counts.indptr[row + 1]
        if start == stop:
            continue                                    # документ без жодного слова
        columns = counts.indices[start:stop]
        times = counts.data[start:stop]
        best_by_tfidf = columns[int(np.argmax(weights.data[start:stop]))]
        leaders = columns[times == times.max()]          # усі слова з максимальною частотою
        if len(leaders) > 1:
            silent += 1                                  # частота не має думки
        else:
            unique += 1
            if best_by_tfidf != leaders[0]:
                overridden += 1
    return silent, unique, overridden

for seed in SEEDS:
    silent, unique, overridden = main_word_disagreement(sample_documents(seed))
    total = silent + unique
    print(f"зерно {seed}: частота мовчить у {silent} із {total} документів "
          f"({100*silent/total:.1f} %)")
    print(f"          там, де вона говорить ({unique} док.), IDF перекриває її "
          f"у {100*overridden/unique:.1f} % випадків")

Ось два числа теми, і вони кажуть різне про одне й те саме.

**Перше.** У 87 із кожних 100 документів частота слів не дає жодної підказки:
всі слова по разу. Там TF-IDF — це фактично **чиста IDF**, і саме вона обирає
головне слово одноосібно.

**Друге.** У решті документів, де частота таки має улюбленця, IDF відбирає в нього
перше місце більш ніж у чотирьох випадках із пʼяти.

Разом: IDF — не косметичний множник. На короткому тексті вона **і є** відповіддю.

## 7 · Своїми руками: жодної магії всередині

Найкорисніша перевірка практики — написати те саме самому й переконатись, що
бібліотека рахує рівно те, що написано у формулі.

Кроки такі:

1. `df(w)` — у скількох документах слово трапилось.
2. `idf(w) = ln((1 + N) / (1 + df(w))) + 1`.
3. Помножити кожну клітинку матриці частот на IDF її колонки.
4. Поділити кожен рядок на його довжину (L2-нормалізація).

In [ ]:
texts = sample_documents(0)
vec, counts = count_matrix(texts)
n_documents = counts.shape[0]

# крок 1: у скількох документах трапилось кожне слово
document_frequency = np.asarray((counts > 0).sum(axis=0)).ravel().astype(float)

# крок 2: обернена документна частота зі згладжуванням, як у sklearn
our_idf = np.log((1 + n_documents) / (1 + document_frequency)) + 1.0

# крок 3: множимо частоти на вагу слова, крок 4: рівняємо довжину рядків
our_tfidf = normalize(counts.multiply(our_idf).tocsr())

# те саме бібліотекою, з тим самим словником
library, library_tfidf = fit_tfidf(texts, vocabulary=vec.vocabulary_)

assert np.allclose(our_idf, library.idf_), "IDF розійшлася!"
assert np.allclose(our_tfidf.toarray()[:200], library_tfidf.toarray()[:200]), "матриця розійшлася!"
print("✅ збігається: максимальна різниця IDF", float(np.max(np.abs(our_idf - library.idf_))))

## 8 · Чому логарифм, а не просто N / df

Ідея IDF — «рідкісне цінніше за часте» — сама по собі логарифма не вимагає.
Найпростіше було б узяти `N / df`. Заміряємо, що з цього вийде.

Міряти будемо так: у кожному документі порахуємо, яку **частку всієї ваги
документа** забирає його найважче слово. Якщо схема справедлива, вага розподілена
між словами; якщо схема зривається — одне слово забирає майже все.

In [ ]:
def top_word_share(counts, idf):
    """Середня частка ваги, яку в документі забирає одне найважче слово."""
    weighted = counts.multiply(idf).tocsr()
    shares = []
    for row in range(weighted.shape[0]):
        start, stop = weighted.indptr[row], weighted.indptr[row + 1]
        if start == stop:
            continue
        row_weights = weighted.data[start:stop]
        shares.append(row_weights.max() / row_weights.sum())
    return float(np.mean(shares))

for seed in SEEDS:
    texts = sample_documents(seed)
    vec, counts = count_matrix(texts)
    n = counts.shape[0]
    df = np.asarray((counts > 0).sum(axis=0)).ravel().astype(float)
    schemes = {
        "без IDF":        np.ones_like(df),
        "log(N/df) + 1":  np.log((1 + n) / (1 + df)) + 1.0,
        "корінь N/df":    np.sqrt(n / df),
        "лінійна N/df":   n / df,
    }
    print(f"зерно {seed}:")
    for name, idf in schemes.items():
        span = idf.max() / idf.min()
        print(f"   {name:<15} частка головного слова {top_word_share(counts, idf):.4f}"
              f"   розмах ваг {span:>8.1f}x")

Число, заради якого все це рахувалось: **без IDF** одне слово забирає 18 % ваги
документа, з логарифмом — 23 %, а з лінійною `N/df` — **59 %**.

Тобто лінійна версія перетворює документ на одне слово: найрідкісніше. Решта тексту
просто перестає впливати. І видно, звідки це береться: розмах ваг у логарифмічної
схеми — приблизно 4.5 рази між найдешевшим і найдорожчим словом, а в лінійної —
понад пʼять тисяч разів.

Логарифм тут не для краси. Він робить рівно одне: **перетворює множення на
додавання**. Слово, у десять разів рідкісніше, стає не в десять разів важливішим,
а на однакову добавку важливішим. Це і є те, чого ми хочемо від «цінності».

## 9 · Згладжування: одиниці зверху й знизу

У формулі `sklearn` є дві одиниці, і вони роблять різні речі.

`+1` у **чисельнику й знаменнику** (`smooth_idf=True`) — це «уявний документ, у
якому є всі слова». Він рятує від ділення на нуль, коли `df = 0`.

Ділення на нуль здається неможливим: як слово може бути у словнику й нізде не
траплятись? Легко — якщо словник узято з іншого корпусу. Заміряємо, наскільки це
часта ситуація.

`+1` **зовні логарифма** — інше: воно не дає вазі впасти в нуль для слова, яке є
геть у всіх документах. Без нього таке слово зникло б із матриці зовсім.

In [ ]:
texts = sample_documents(0)
vec, counts = count_matrix(texts)
n = counts.shape[0]
df = np.asarray((counts > 0).sum(axis=0)).ravel().astype(float)

smoothed = np.log((1 + n) / (1 + df)) + 1.0
plain    = np.log(n / df) + 1.0

print(f"максимальний IDF: зі згладжуванням {smoothed.max():.4f}, без {plain.max():.4f}")
print(f"мінімальний IDF:  зі згладжуванням {smoothed.min():.4f}, без {plain.min():.4f}")
print(f"середня різниця по словнику {np.mean(np.abs(smoothed - plain)):.4f}, "
      f"найбільша {np.max(np.abs(smoothed - plain)):.4f}")
print(f"слів, що трапились рівно в одному документі: {100*np.mean(df == 1):.1f} % словника")

# а тепер — словник із іншої вибірки того самого корпусу
other_vec, _ = count_matrix(sample_documents(1))
unseen = [w for w in other_vec.vocabulary_ if w not in vec.vocabulary_]
print()
print(f"слів чужого словника, яких у цій вибірці немає: {len(unseen)} із "
      f"{len(other_vec.vocabulary_)} ({100*len(unseen)/len(other_vec.vocabulary_):.1f} %)")
print("для них df = 0: без згладжування це ln(N/0) = нескінченність,")
print(f"зі згладжуванням — скінченні ln({n+1}/1) + 1 = {math.log((n+1)/1)+1:.4f}")

Тридцять один відсоток. Тобто якщо ти навчив `TfidfVectorizer` на одній вибірці й
приніс словник на іншу, майже третина слів матиме `df = 0`. Без згладжування це
не «трохи неточно», а `inf` у матриці й `NaN` після нормалізації.

Друге, що робить згладжування, — трохи підрізає верх: 10.90 стає 10.21. На пошук
це, як ми зараз побачимо, майже не впливає, і чесно про це сказати важливіше, ніж
вигадати різницю.

## 10 · Нормалізація: чому довгий документ не має вигравати

Скалярний добуток запиту й документа росте разом із документом: що більше слів,
то більше доданків. Без нормалізації пошук перетворюється на «покажи найдовші
тексти, у яких трапилось потрібне слово».

L2-нормалізація ділить вектор документа на його довжину. Після цього довжина
кожного вектора — рівно одиниця, а скалярний добуток стає **косинусом кута**:
мірою напрямку, а не розміру.

Заміряємо на пошуку. Задача чесна й перевірювана: беремо документ, робимо з нього
запит із трьох слів і дивимось, на якому місці знайдеться сам документ.

In [ ]:
def build_queries(counts, seed, n_queries=300, mode="mixed"):
    """З кожного документа-мішені робимо запит із трьох його слів.
    mode='mixed'  — два найчастіші в корпусі слова документа плюс одне найрідкісніше
                    (так виглядає справжній запит: службові слова плюс змістовне);
    mode='random' — три випадкові слова документа."""
    rng = np.random.default_rng(1000 + seed)
    df = np.asarray((counts > 0).sum(axis=0)).ravel()
    lengths = np.diff(counts.indptr)
    targets = rng.choice(np.where(lengths >= 6)[0], n_queries, replace=False)
    rows, columns = [], []
    for number, target in enumerate(targets):
        start, stop = counts.indptr[target], counts.indptr[target + 1]
        terms = counts.indices[start:stop]
        if mode == "mixed":
            by_df = terms[np.argsort(-df[terms])]
            picked = [by_df[0], by_df[1], by_df[-1]]
        else:
            picked = rng.choice(terms, 3, replace=False)
        for term in picked:
            rows.append(number)
            columns.append(term)
    queries = sp.csr_matrix((np.ones(len(rows)), (rows, columns)),
                            shape=(n_queries, counts.shape[1]))
    return queries, targets

def search_quality(counts, queries, targets, idf, use_l2):
    documents_matrix = counts.multiply(idf).tocsr()
    queries_matrix = queries.multiply(idf).tocsr()
    if use_l2:
        documents_matrix = normalize(documents_matrix)
        queries_matrix = normalize(queries_matrix)
    scores = (queries_matrix @ documents_matrix.T).toarray()
    own = scores[np.arange(len(targets)), targets]
    # місце мішені: скільки документів набрало більше (нічиї ріжемо за номером)
    better = (scores > own[:, None]).sum(axis=1)
    ties_before = ((scores == own[:, None]) &
                   (np.arange(counts.shape[0])[None, :] < targets[:, None])).sum(axis=1)
    place = 1 + better + ties_before
    lengths = np.diff(counts.indptr)
    top_ten = np.argsort(-scores, axis=1)[:, :10]
    return dict(mrr=float(np.mean(1 / place)),
                first=float(np.mean(place == 1)),
                top_length=float(lengths[top_ten].mean()))

for seed in SEEDS:
    texts = sample_documents(seed)
    vec, counts = count_matrix(texts)
    n = counts.shape[0]
    df = np.asarray((counts > 0).sum(axis=0)).ravel().astype(float)
    idf = np.log((1 + n) / (1 + df)) + 1.0
    queries, targets = build_queries(counts, seed)
    lengths = np.diff(counts.indptr)
    print(f"зерно {seed}: середня довжина документа {lengths.mean():.2f} слова")
    for use_l2 in (False, True):
        r = search_quality(counts, queries, targets, idf, use_l2)
        label = "з L2" if use_l2 else "без L2"
        print(f"   {label:<6} MRR {r['mrr']:.4f}  знайдено першим {r['first']:.4f}  "
              f"середня довжина топ-10: {r['top_length']:.2f} слова")

Середній документ корпусу — вісім слів. Без нормалізації пошук піднімає нагору
документи по 52-64 слова, тобто **всемеро довші за середній**, і знаходить те, що
просили, першим приблизно в 41 % випадків замість 75 %. Це не тонке налаштування,
а умова, щоб пошук узагалі працював.

## 11 · Пошук: де різниця видима, а де ні

Тепер порівняємо самі схеми ваг на тій самій задачі. І — важливо — на **двох різних
типах запитів**, бо відповідь від них залежить.

In [ ]:
def compare_schemes(mode):
    collected = collections.defaultdict(list)
    for seed in SEEDS:
        texts = sample_documents(seed)
        vec, counts = count_matrix(texts)
        n = counts.shape[0]
        df = np.asarray((counts > 0).sum(axis=0)).ravel().astype(float)
        queries, targets = build_queries(counts, seed, mode=mode)
        schemes = {
            "без IDF":      np.ones_like(df),
            "TF-IDF (log)": np.log((1 + n) / (1 + df)) + 1.0,
            "без згладж.":  np.log(n / df) + 1.0,
            "лінійна N/df": n / df,
        }
        for name, idf in schemes.items():
            collected[name].append(search_quality(counts, queries, targets, idf, True))
    print(f"запити типу «{mode}»:")
    for name, runs in collected.items():
        mrr = [r["mrr"] for r in runs]
        print(f"   {name:<14} MRR {np.mean(mrr):.4f}  розкид по зернах {max(mrr)-min(mrr):.4f}"
              f"   знайдено першим {np.mean([r['first'] for r in runs]):.4f}")
    base = [r["mrr"] for r in collected["без IDF"]]
    best = [r["mrr"] for r in collected["TF-IDF (log)"]]
    gains = [b - a for b, a in zip(best, base)]
    print(f"   виграш TF-IDF над частотою по зернах: "
          + ", ".join(f"{g:+.4f}" for g in gains))
    return collected

mixed = compare_schemes("mixed")
print()
random_terms = compare_schemes("random")

Три висновки, і один із них незручний.

**Перший.** На запиті «два звичні слова плюс одне змістовне» — тобто на тому, як
люди справді пишуть запити, — TF-IDF виграє в частоти близько **0.19 MRR** при
розкиді по зернах близько 0.03. Виграш у шість разів більший за шум, тобто справжній.

**Другий.** Лінійна `N/df` програє логарифмічній помітно й стабільно. Те саме, що
показала частка ваги: коли одне слово забирає все, пошук ламається.

**Третій, незручний.** На запиті з трьох випадкових слів документа різниця між
TF-IDF і простою частотою **менша за розкид між зернами**. Тобто на таких запитах
TF-IDF не виграє нічого, і чесно це так і назвати. Причина видна з першого досліду:
випадкові слова короткого документа й так переважно рідкісні, і зважувати там нічого.

TF-IDF потрібна не «завжди», а тоді, коли **в запиті поруч стоять часте й рідкісне
слово**. Тоді вона й вирішує, яке з них слухати.

Подивимось на це очима — на конкретних запитах до тих самих 20 000 документів.

In [ ]:
texts = sample_documents(0)
vec, counts = count_matrix(texts)
n = counts.shape[0]
df = np.asarray((counts > 0).sum(axis=0)).ravel().astype(float)
idf = np.log((1 + n) / (1 + df)) + 1.0
by_tfidf = normalize(counts.multiply(idf).tocsr())
by_count = normalize(counts.astype(float))
words = vec.get_feature_names_out()

def show(query, how_many=3):
    q = vec.transform([query])
    print("ЗАПИТ:", query)
    print("  ваги слів запиту:",
          ", ".join(f"{words[j]} idf {idf[j]:.2f}" for j in q.indices))
    for name, matrix, weighted in (("за частотою", by_count, False),
                                   ("за TF-IDF  ", by_tfidf, True)):
        qq = normalize(q.multiply(idf).tocsr()) if weighted else normalize(q.astype(float))
        scores = np.asarray((qq @ matrix.T).todense()).ravel()
        best = np.argsort(-scores)[:how_many]
        print(f"  {name}:")
        for j in best:
            print(f"     {scores[j]:.3f}  {' '.join(texts[j].split())[:58]}")
    print()

show("дані про принтер")
show("не вдалося прочитати каталог")

Перший запит — різниця видима одразу: за частотою нагору вилазить документ
«Дані:» (два слова, одне з них із запиту — і в короткому документі це дає високий
косинус), а TF-IDF ставить першим «Отримання інформації про принтер». Слово
«принтер» рідкісне, і воно тягне.

Другий запит — різниця не видима зовсім: обидві схеми дають ту саму трійку.
Коли в запиті всі слова однаково рідкісні, зважувати нічого.

## 12 · Де TF-IDF ламається · випадок 1: синоніми

Тут нам щастить із корпусом. Той самий англійський рядок різні перекладачі
переклали по-різному — і ми маємо пари документів, про які **точно** відомо, що
вони означають одне й те саме, бо походять з одного оригіналу.

Це ідеальна перевірка. Якщо TF-IDF розуміє зміст, косинус таких пар має бути високим.

In [ ]:
NOISE = re.compile(r"[@<>]|https?://")

def paraphrase_pairs():
    """Пари українських перекладів того самого англійського рядка,
    у яких майже немає спільних слів."""
    by_source = collections.defaultdict(dict)
    for program, source, target in corpus:
        if source.strip() == 'translator-credits':
            continue
        by_source[source][' '.join(clean(target).split())] = program
    found = []
    for source, variants in by_source.items():
        texts = list(variants)
        for i in range(len(texts)):
            for j in range(i + 1, len(texts)):
                first, second = texts[i], texts[j]
                if NOISE.search(first) or NOISE.search(second):
                    continue
                a = set(tokenize(first.lower()))
                b = set(tokenize(second.lower()))
                if len(a) < 3 or len(b) < 3:
                    continue
                overlap = len(a & b) / len(a | b)
                if overlap < 0.34:
                    found.append((first, second, len(a & b)))
    return found

pairs = paraphrase_pairs()
print(f"пар «те саме іншими словами»: {len(pairs)}, "
      f"з них без жодного спільного слова: {sum(1 for p in pairs if p[2] == 0)}")
print()
for first, second, shared in pairs[:4]:
    print(f"  спільних слів {shared}")
    print(f"    A: {first}")
    print(f"    B: {second}")

In [ ]:
for seed in SEEDS:
    texts = sample_documents(seed)
    tfidf, _ = fit_tfidf(texts)
    left = tfidf.transform([p[0] for p in pairs])
    right = tfidf.transform([p[1] for p in pairs])
    cosine = np.asarray(left.multiply(right).sum(axis=1)).ravel()

    # контроль: випадкові пари документів корпусу, які нічого спільного не мають
    rng = np.random.default_rng(7 + seed)
    a = tfidf.transform([texts[i] for i in rng.choice(len(texts), len(pairs))])
    b = tfidf.transform([texts[i] for i in rng.choice(len(texts), len(pairs))])
    random_cosine = np.asarray(a.multiply(b).sum(axis=1)).ravel()

    # для порівняння: ті самі пари, але подані символьними триграмами
    char_tfidf = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 3))
    char_tfidf.fit(texts)
    ca = char_tfidf.transform([p[0] for p in pairs])
    cb = char_tfidf.transform([p[1] for p in pairs])
    char_cosine = np.asarray(ca.multiply(cb).sum(axis=1)).ravel()

    print(f"зерно {seed}: синоніми — середній косинус {cosine.mean():.4f}, "
          f"медіана {np.median(cosine):.4f}, рівно нуль у {100*np.mean(cosine == 0):.1f} %")
    print(f"          випадкові пари — {random_cosine.mean():.4f}; "
          f"символьні триграми на тих самих парах — {char_cosine.mean():.4f}")

Ось і поломка, і вона повна. Пари документів, які означають **буквально одне й те
саме**, дістають середній косинус близько 0.24, а майже кожна пʼята з них —
**рівно нуль**, тобто рівно стільки, скільки два випадкові документи корпусу.

Причина не в налаштуваннях. TF-IDF порівнює **написання**, а не зміст: слова
«регулярний» і «формальний» для неї — дві незалежні колонки без жодного звʼязку.
Ніяким підбором ваг це не лікується, бо зміст у поданні просто відсутній.

Символьні триграми ловлять більше (0.46 проти 0.24) — але лише тому, що вони
бачать спільні корені й закінчення, а не тому, що розуміють синонімію. Справжня
відповідь на це питання — блок 3 курсу, де слово стане вектором
([ембединги, тема 12](../12-word2vec/lecture.html)).

## 13 · Випадок 2: дуже короткі документи

Другий спосіб зламати TF-IDF — дати їй короткий текст. Заміряємо дві речі:
яку частку довжини вектора несе одне слово, і наскільки вектор змінюється, якщо
одне слово прибрати.

In [ ]:
BUCKETS = [(1, 3), (4, 6), (7, 10), (11, 20), (21, 10**6)]

for seed in SEEDS:
    texts = sample_documents(seed)
    vec, counts = count_matrix(texts)
    n = counts.shape[0]
    df = np.asarray((counts > 0).sum(axis=0)).ravel().astype(float)
    idf = np.log((1 + n) / (1 + df)) + 1.0
    unit = normalize(counts.multiply(idf).tocsr())     # рядки одиничної довжини
    lengths = np.asarray(counts.sum(axis=1)).ravel().astype(int)
    rng = np.random.default_rng(50 + seed)

    biggest, after_loss = [], []
    for row in range(unit.shape[0]):
        start, stop = unit.indptr[row], unit.indptr[row + 1]
        if start == stop:
            biggest.append(np.nan); after_loss.append(np.nan); continue
        row_weights = unit.data[start:stop]
        biggest.append(row_weights.max())               # вектор одиничний, тож це і є частка
        if stop - start == 1:
            after_loss.append(0.0); continue
        damaged = row_weights.copy()
        damaged[rng.integers(0, stop - start)] = 0      # прибираємо одне слово
        after_loss.append(float(row_weights @ damaged / np.linalg.norm(damaged)))
    biggest = np.array(biggest); after_loss = np.array(after_loss)

    print(f"зерно {seed}:")
    for low, high in BUCKETS:
        mask = (lengths >= low) & (lengths <= high) & np.isfinite(biggest)
        name = f"{low}-{high}" if high < 10**6 else "21+"
        print(f"   {name:<6} слів: {mask.sum():>5} док. ({100*mask.mean():>4.1f} %) | "
              f"головне слово несе {biggest[mask].mean():.4f} довжини вектора | "
              f"косинус із собою без одного слова {np.nanmean(after_loss[mask]):.4f}")

У документі на три слова одне слово несе **72 %** довжини вектора, і втрата одного
слова роняє схожість із самим собою до 0.75. У документі на 21 слово й більше —
40 % і 0.98.

Практичний наслідок: на коротких текстах TF-IDF **дуже чутлива до випадковості**.
Одна інша словоформа, один синонім, одна одруківка — і документ поїхав. А наш
корпус саме такий: половина документів має від чотирьох до шести слів.

## 14 · Випадок 3: рідкісне в корпусі — не те саме, що рідкісне в мові

IDF не знає мови. Вона знає рівно один корпус — той, на якому її порахували.
Слово, якого в цьому корпусі мало, дістане велику вагу, навіть якщо в житті
його знає кожна дитина.

Перевіримо просто: візьмемо звичайнісінькі українські слова й подивимось на їхню
`df` у нашому корпусі.

In [ ]:
texts = sample_documents(0)
vec, counts = count_matrix(texts)
n = counts.shape[0]
df = np.asarray((counts > 0).sum(axis=0)).ravel().astype(float)
idf = np.log((1 + n) / (1 + df)) + 1.0

everyday = ['мама', 'хліб', 'дощ', 'сонце', 'дерево', 'кіт', 'любов', 'вулиця', 'пісня', 'брат']
technical = ['налаштування', 'файл', 'помилка', 'каталог', 'сервер']

print("побутові українські слова:")
for word in everyday:
    if word in vec.vocabulary_:
        j = vec.vocabulary_[word]
        print(f"   {word:<10} df={int(df[j]):<5} idf={idf[j]:.4f}")
    else:
        print(f"   {word:<10} у корпусі немає взагалі")
print()
print("технічні слова:")
for word in technical:
    j = vec.vocabulary_[word]
    print(f"   {word:<14} df={int(df[j]):<5} idf={idf[j]:.4f}")

Дев'ять із десяти побутових слів у корпусі **не трапляються жодного разу**. Єдине,
що прорвалось, — «дерево», і то не рослина, а елемент інтерфейсу; воно дістає
IDF 8.01, тобто важить **більше за «сервер»**.

Тепер порахуємо це не на десяти словах, а систематично. Візьмемо одну програму з
власним доменом — векторний редактор Inkscape — і порівняємо IDF її слів,
порахований усередині цього домену, з IDF, порахованим на решті корпусу.

In [ ]:
inkscape = [clean(target) for program, _, target in corpus if program == 'inkscape']
outside_all = [clean(target) for program, _, target in corpus if program != 'inkscape']

if len(inkscape) < 200:
    print("у цьому корпусі немає Inkscape — розділ пропущено")
else:
    rng = np.random.default_rng(0)
    outside = [outside_all[i] for i in rng.choice(len(outside_all), N_DOCS, replace=False)]
    inside_vec, inside_counts = count_matrix(inkscape)
    outside_vec, outside_counts = count_matrix(outside)

    def idf_of(counts):
        n = counts.shape[0]
        df = np.asarray((counts > 0).sum(axis=0)).ravel().astype(float)
        return np.log((1 + n) / (1 + df)) + 1.0, df

    inside_idf, inside_df = idf_of(inside_counts)
    outside_idf, outside_df = idf_of(outside_counts)
    missing_weight = math.log((N_DOCS + 1) / 1) + 1      # слова, якого зовні немає взагалі

    uses = np.asarray(inside_counts.sum(axis=0)).ravel()
    words = inside_vec.get_feature_names_out()
    top = np.argsort(-uses)[:400]

    rows = []
    for j in top:
        word = words[j]
        if word in outside_vec.vocabulary_:
            outer = float(outside_idf[outside_vec.vocabulary_[word]])
        else:
            outer = missing_weight
        rows.append((word, int(uses[j]), float(inside_idf[j]), outer, outer - float(inside_idf[j])))

    gaps = [r[4] for r in rows]
    print(f"документів Inkscape: {len(inkscape)}")
    print(f"серед 400 найуживаніших слів Inkscape середній розрив IDF "
          f"«зовні мінус усередині»: {np.mean(gaps):.4f}")
    print()
    rows.sort(key=lambda r: -r[4])
    for word, times, inner, outer, gap in rows[:8]:
        print(f"   {word:<14} ужитків у Inkscape {times:>4} | IDF свій {inner:.4f} | "
              f"IDF чужий {outer:.4f} | розрив {gap:+.4f}")

Слово «контур» — звичайне українське слово й буденний термін у векторному
редакторі — дістає від зовнішнього корпусу **максимальну можливу вагу**, бо там
воно не трапляється взагалі. Середній розрив по чотирьохстах найуживаніших
словах домену — близько 1.9 одиниці IDF.

Це і є причина, чому IDF **не можна переносити між доменами**, і чому «стоп-слова»
корисно рахувати самим, а не брати готовий список: у нашому корпусі «вдалося» —
службове слово, а в корпусі новин воно нормальне.

## 15 · Підсумок блоку 1 в числах

Пʼять тем — одна лінія: **текст не таблиця → ріжемо на шматки → зводимо форми →
рахуємо → зважуємо**. Порахуємо тут головні числа кожного кроку, щоб побачити їх
поруч.

In [ ]:
t0 = time.time()

# ── тема 01: закони природної мови ──────────────────────────────────
ranks = np.arange(1, len(frequency) + 1)
freqs = np.array(sorted(frequency.values(), reverse=True), dtype=float)
window = ranks <= 10000
zipf_slope = np.polyfit(np.log(ranks[window]), np.log(freqs[window]), 1)[0]
# нахил дуже чутливий до вікна підгонки: на рангах 1-1000 виходить -0.85,
# на всьому хвості -1.39. Тому вікно називаємо явно.
hapax = 100 * np.mean(freqs == 1)

seen, total_tokens, xs, ys = set(), 0, [], []
for index, text in enumerate(documents):
    for word in tokenize(text.lower()):
        seen.add(word)
        total_tokens += 1
    if (index + 1) % 2000 == 0:
        xs.append(total_tokens); ys.append(len(seen))
heaps_beta = np.polyfit(np.log(xs), np.log(ys), 1)[0]

print(f"01 · нахил Ципфа на рангах 1-10000: {zipf_slope:.4f} (природна мова близько -1)")
print(f"01 · слів, що трапились рівно раз: {hapax:.1f} %")
print(f"01 · показник Гіпса beta = {heaps_beta:.4f} (словник росте й не насичується)")

# ── тема 02: та сама думка двома мовами ─────────────────────────────
ukrainian_chars = sum(len(clean(target)) for _, _, target in corpus)
english_chars = sum(len(clean(source)) for _, source, _ in corpus)
if english_chars > 0:
    print(f"02 · той самий зміст українською довший у символах у "
          f"{ukrainian_chars / english_chars:.4f} раза")

# ── тема 03: скільки форм в однієї леми ─────────────────────────────
cyrillic = [w for w in frequency if re.fullmatch('[а-яіїєґʼ]+', w)]
try:
    import pymorphy3
    analyzer = pymorphy3.MorphAnalyzer(lang='uk')
    lemmas = set(analyzer.parse(word)[0].normal_form for word in cyrillic)
    print(f"03 · словоформ {len(cyrillic)}, лем {len(lemmas)}, "
          f"падіння словника {100 * (1 - len(lemmas) / len(cyrillic)):.1f} %")
except Exception as error:
    print("03 · pymorphy3 недоступний:", error)

# ── тема 04: розрідженість ──────────────────────────────────────────
vec, counts = count_matrix(sample_documents(0))
cells = counts.shape[0] * counts.shape[1]
dense_gb = cells * 8 / 1024**3
sparse_mb = (counts.data.nbytes + counts.indices.nbytes + counts.indptr.nbytes) / 1024**2
print(f"04 · матриця {counts.shape[0]} x {counts.shape[1]}, заповнено "
      f"{100*counts.nnz/cells:.4f} %")
print(f"04 · щільно це {dense_gb:.2f} ГБ, розріджено {sparse_mb:.2f} МБ "
      f"({dense_gb*1024/sparse_mb:.0f} разів різниці)")

# ── тема 05: що додала вага ─────────────────────────────────────────
silent, unique, overridden = main_word_disagreement(sample_documents(0))
print(f"05 · частота мовчить у {100*silent/(silent+unique):.1f} % документів; "
      f"де говорить — IDF перекриває {100*overridden/unique:.1f} %")
print()
print(f"(розділ рахувався {time.time() - t0:.1f} с)")

Прочитати цю таблицю варто так.

**01.** Мова підпорядкована степеневим законам: нахил Ципфа близько −1, третина
слів трапляється рівно раз, словник росте як корінь із тексту й не насичується.
Саме тому «просто перелічити всі слова» не працює ніколи.

**02.** Той самий зміст українською довший, і токенізатор ріже його на більше
шматків. Але ріже він за доменом, а не за мовою — і TF-IDF успадкувала цю
властивість цілком: «вдалося» дешеве саме тому, що корпус про помилки.

**03.** Зведення форм до леми ріже словник більш ніж удвічі. Для TF-IDF це прямо
корисно: без лематизації «файл», «файла» й «файлу» — три незалежні колонки, і
кожна отримує свою IDF, розмазуючи вагу того самого поняття на три частини.

**04.** Матриця розріджена до чотирьох сотих відсотка. TF-IDF нічого тут не міняє:
вона переписує значення в тих самих клітинках.

**05.** І нарешті вага. Вона робить велику роботу — міняє головне слово в більшості
документів, дає 0.19 MRR на реалістичних запитах — і не робить ніякої на трьох
випадкових словах. І вона не бачить синонімів узагалі.

Це і є межа блоку 1. Усе, що ми будували пʼять тем, тримається на **збігу
написань**. Наступний блок навчиться на цьому поданні розвʼязувати справжні задачі —
класифікацію, пошук, тематичне моделювання, — а третій нарешті замінить збіг
написань на схожість змісту.

## 16 · Дані, на яких працюють фігури лекції

Фігури в лекції рахують TF-IDF просто в браузері — на 120 справжніх документах
нашого корпусу й на справжніх `df`, узятих із тих самих 20 000. Ця клітинка
друкує числа, які там показано, щоб їх можна було звірити.

In [ ]:
allowed = re.compile(r"^[А-Яа-яІіЇїЄєҐґʼ ,.:\-]+$")
texts = sample_documents(0)
unique_texts = {}
for text in texts:
    tidy = ' '.join(text.split())
    if 4 <= len(tokenize(tidy.lower())) <= 8 and len(tidy) <= 58 \
       and allowed.fullmatch(tidy) and tidy.lower() not in unique_texts:
        unique_texts[tidy.lower()] = tidy
selected = list(unique_texts.values())
mini = selected[::max(1, len(selected) // 120)][:120]

vec, counts = count_matrix(texts)
n = counts.shape[0]
df = np.asarray((counts > 0).sum(axis=0)).ravel().astype(float)

def mini_share(kind):
    shares = []
    for text in mini:
        seen = collections.Counter(w for w in tokenize(text.lower()) if w in vec.vocabulary_)
        weights = []
        for word, times in seen.items():
            d = df[vec.vocabulary_[word]]
            if kind == 'none':
                weight = 1.0
            elif kind == 'log':
                weight = math.log((1 + n) / (1 + d)) + 1
            else:
                weight = n / d
            weights.append(times * weight)
        shares.append(max(weights) / sum(weights))
    return float(np.mean(shares))

print("мінікорпус фігур:", len(mini), "документів")
for kind, name in (('none', 'без IDF'), ('log', 'log(N/df)+1'), ('lin', 'лінійна N/df')):
    print(f"   частка головного слова, {name:<13} {mini_share(kind):.4f}")

counts_per_doc = [collections.Counter(tokenize(t.lower())) for t in mini]
with_opinion = sum(1 for c in counts_per_doc
                   if sum(1 for v in c.values() if v == max(c.values())) == 1)
print(f"   документів, де частота має унікального улюбленця: {with_opinion} зі {len(mini)}")
print()
print("перші пʼять документів мінікорпусу:")
for text in mini[:5]:
    print("   ", text)

## Завдання

### 🟢 Рівень 1

Порахуй IDF на **своєму** тексті: візьми будь-які 200-500 коротких документів
(листи, назви товарів, повідомлення в чаті — що завгодно) і надрукуй десять слів
із найменшою IDF і десять із найбільшою.

**Зроблено, якщо:** ти можеш пояснити словами, чому саме ці слова опинились унизу
списку, і назвати хоч одне, яке потрапило туди через **домен**, а не через мову.

### 🟡 Рівень 2

Повтори замір із розділу 8 (частка ваги головного слова) для схеми
`idf = (N/df) ** p` при `p` = 0, 0.25, 0.5, 0.75, 1. Побудуй графік «p → частка».

**Зроблено, якщо:** на графіку видно, при якому `p` частка починає різко зростати,
і ти можеш сказати, чому логарифм поводиться як маленьке `p`, а не як `p = 1`.

### 🔴 Рівень 3

Реалізуй **BM25** (формула є в лекції) і порівняй його з TF-IDF на задачі з
розділу 10-11: три зерна, обидва типи запитів, метрика MRR.

**Зроблено, якщо:** ти назвав виграш або програш у MRR **разом із розкидом по
зернах** і сказав прямо, чи більший цей виграш за розкид. Відповідь «різниці немає»
є повноцінною відповіддю, якщо вона підкріплена числами.